# Анализ эксперимента `stab_4`

Чтение данных из `data/archive/05_dataset/raw/stab_4.csv` и построение графика `Напряжение` от времени.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

csv_path = Path("..") / "data" / "archive" / "05_dataset" / "raw" / "stab_4.csv"

# Оставляем строку с названиями колонок, пропускаем строки с единицами/служебными подписями.
raw_df = pd.read_csv(csv_path, skiprows=[1, 2], low_memory=False)
raw_df.columns = raw_df.columns.str.replace("\ufeff", "", regex=False).str.strip()

raw_df["Время"] = pd.to_numeric(raw_df["Время"], errors="coerce")
raw_df["Напряжение"] = pd.to_numeric(raw_df["Напряжение"], errors="coerce")
raw_df["ЦАП"] = pd.to_numeric(raw_df["ЦАП"], errors="coerce")
raw_df = raw_df.dropna(subset=["Время", "Напряжение", "ЦАП"]).copy()

# Перевод в отсчеты АЦП/ЦАП для отчета.
raw_df["АЦП"] = raw_df["Напряжение"] / 1.25 * 1023
raw_df["ЦАП_отсчеты"] = raw_df["ЦАП"] / 5.0 * 4095

# Время эксперимента от нуля.
raw_df["Время_с_0"] = raw_df["Время"] - raw_df["Время"].min()

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

setpoint = 120

fig, ax1 = plt.subplots(figsize=(14, 7))

# Первая ось (отсчеты АЦП фотодетектора)
ax1.plot(
    raw_df["Время_с_0"],
    raw_df["АЦП"],
    color="blue",
    alpha=0.7,
    linewidth=0.8,
)
ax1.axhline(y=setpoint, color="black", linestyle="--", linewidth=5)

ax1.set_xlabel("Время, с")
ax1.set_ylabel("Напряжение на фотодетекторе", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")

# Вторая ось (отсчеты ЦАП на фазовращателе)
ax2 = ax1.twinx()
ax2.plot(
    raw_df["Время_с_0"],
    raw_df["ЦАП_отсчеты"],
    color="red",
    alpha=0.7,
    linewidth=0.8,
)

ax2.set_ylabel("Напряжение на фазовращателе", color="red")
ax2.tick_params(axis="y", labelcolor="red")

plt.grid(False)
plt.tight_layout()
plt.savefig("combined_plot.png", dpi=300)
plt.show()
